# LAB_ML02 — Solución guiada
## Redes neuronales con PyTorch

Mismo problema de churn, ahora con una red `11 → 32 → 16 → 1`.

## 0. Comprobar PyTorch

PyTorch debe estar instalado como librería del Compute.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

## 1. Cargar los datos

In [ ]:
from pyspark.sql import functions as F

TRAIN_PATH = "/Volumes/training/shared/source/ml/customer_churn_gold.csv"
SCORING_PATH = "/Volumes/training/shared/source/ml/customer_churn_scoring.csv"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(TRAIN_PATH)
)

display(df.limit(10))
df.printSchema()
print("Filas:", df.count())

display(
    df.groupBy("churn")
      .count()
      .orderBy("churn")
)

## 2. Recuperar las 11 features numéricas

In [ ]:
numeric_types = {"byte", "short", "int", "bigint", "long", "float", "double", "decimal"}

feature_cols = [
    field.name
    for field in df.schema.fields
    if field.name not in {"customer_id", "churn"}
    and (
        field.dataType.simpleString() in numeric_types
        or field.dataType.simpleString().startswith("decimal")
    )
]

print("Features:", feature_cols)
print("Número de features:", len(feature_cols))

assert len(feature_cols) == 11, (
    f"Se esperaban 11 features numéricas y se encontraron {len(feature_cols)}"
)

In [ ]:
pdf = df.select("customer_id", *feature_cols, "churn").toPandas()

X = pdf[feature_cols].astype("float32")
y = pdf["churn"].astype("float32")

## 3. Separar train, validation y test

Primero reservamos test; después separamos validation dentro de train.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## 4. Normalización

El scaler aprende únicamente de train.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

## 5. Convertir a tensores

In [ ]:
import numpy as np

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1, 1)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val.to_numpy(), dtype=torch.float32).view(-1, 1)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1, 1)

print(X_train_t.shape, y_train_t.shape)

## 6. TensorDataset y DataLoader

El DataLoader entrega mini-batches y puede barajar train en cada epoch.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 128

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

len(train_loader)

## 7. Definir la red

La última capa devuelve logits; `BCEWithLogitsLoss` aplica internamente la formulación estable de sigmoid + binary cross entropy.

In [ ]:
from torch import nn

class ChurnNet(nn.Module):
    def __init__(self, n_features=11):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

model = ChurnNet(n_features=len(feature_cols))
print(model)

## 8. Loss y optimizer

In [ ]:
LEARNING_RATE = 0.001
EPOCHS = 20

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

## 9. Función de evaluación de loss

In [ ]:
def evaluate_loss(model, X_tensor, y_tensor):
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        loss = criterion(logits, y_tensor)
    return float(loss.item())

## 10. Entrenamiento

En cada batch seguimos el ciclo `zero_grad → forward → loss → backward → step`.

In [ ]:
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        batch_size = X_batch.size(0)
        running_loss += loss.item() * batch_size
        seen += batch_size

    train_loss = running_loss / seen
    val_loss = evaluate_loss(model, X_val_t, y_val_t)

    history.append((epoch, train_loss, val_loss))

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f}"
    )

### Overfitting

Si train loss continúa bajando mientras validation loss empeora de forma sostenida, el modelo está ajustándose cada vez más al conjunto de entrenamiento sin mejorar su capacidad de generalización.

## 11. Evaluar en test

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

model.eval()

with torch.no_grad():
    test_logits = model(X_test_t)
    test_proba = torch.sigmoid(test_logits).cpu().numpy().ravel()

test_pred = (test_proba >= 0.5).astype(int)
y_test_np = y_test.to_numpy().astype(int)

metrics = {
    "accuracy": accuracy_score(y_test_np, test_pred),
    "precision": precision_score(y_test_np, test_pred, zero_division=0),
    "recall": recall_score(y_test_np, test_pred, zero_division=0),
    "f1": f1_score(y_test_np, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test_np, test_proba),
}

print(metrics)
print("\nMatriz de confusión:")
print(confusion_matrix(y_test_np, test_pred))
print("\nClassification report:")
print(classification_report(y_test_np, test_pred, zero_division=0))

## 12. Registrar con MLflow

Se registra el modelo PyTorch sin parámetros de serialización específicos, para mantener compatibilidad con la versión de MLflow del entorno.

In [ ]:
import mlflow
import mlflow.pytorch

with mlflow.start_run(run_name="LAB_ML02_PyTorch") as run:
    mlflow.log_param("architecture", "11-32-16-1")
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("features", ",".join(feature_cols))

    for name, value in metrics.items():
        mlflow.log_metric(name, float(value))

    mlflow.pytorch.log_model(model, "model")

    run_id = run.info.run_id

print("MLflow Run ID:", run_id)

## 13. Cargar los clientes para scoring

In [ ]:
scoring_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SCORING_PATH)
)

print("Filas scoring:", scoring_df.count())
display(scoring_df.limit(10))

## 14. Inferencia con la red

Aplicamos **el mismo scaler** entrenado exclusivamente con train.

In [ ]:
scoring_pdf = scoring_df.select("customer_id", *feature_cols).toPandas()

X_scoring = scoring_pdf[feature_cols].astype("float32")
X_scoring_scaled = scaler.transform(X_scoring)

X_scoring_t = torch.tensor(
    X_scoring_scaled,
    dtype=torch.float32
)

model.eval()

with torch.no_grad():
    scoring_logits = model(X_scoring_t)
    scoring_proba = torch.sigmoid(scoring_logits).cpu().numpy().ravel()

scoring_pdf["predicted_churn"] = (scoring_proba >= 0.5).astype(int)
scoring_pdf["churn_probability"] = scoring_proba

scored_spark = spark.createDataFrame(
    scoring_pdf[["customer_id", "predicted_churn", "churn_probability"]]
)

display(
    scored_spark.orderBy(F.desc("churn_probability"))
)

## 15. Resultado

La estructura general es la misma que en LAB_ML01: preparar datos, entrenar solo con train, evaluar con datos no vistos, registrar el experimento y reutilizar exactamente el mismo preprocesamiento para inferencia.